In [ ]:
# Author: Filippo Zinetti
# April 2026
# Basic simulation of a 1D reflected slab reactor with two neutron groups.

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

# Helper functions

In [ ]:
def iterate_sol(phi, A, S, max_iter=10, R=0, nu=1, Sigma_f=1):
	epsilon = 1
	k_eff_old = 0
	max_iter = 40
	i = 0

	while epsilon > 0.001 and i < max_iter:
		i += 1

		k_eff = np.sum(get_S(phi, R=R, nu=nu, Sigma_f=Sigma_f)) / np.sum(S)
		
		epsilon = np.abs(k_eff-k_eff_old)
		k_eff_old = k_eff
		
		S = get_S(phi, R=R, nu=nu, Sigma_f=Sigma_f)
		b = S / k_eff
		phi = spsolve(A, b)
		print(f"k_eff: {k_eff:.5f}, eps: {epsilon:.5f}")

		max = np.max(phi)
		phi /= max

	return phi

def get_fast_S(f, R=0, nu=1, Sigma_f=1):
	f_bounded = f
	if R != 0:
		f_bounded = f[R:-R]
	return np.concat((np.zeros(R), nu * Sigma_f * f_bounded, np.zeros(R)))
	
def get_thermal_S(f, R=0, Sigma_sFT=None):
	return Sigma_sFT * f
	
def get_S(f, R=0, nu=1, Sigma_f=1):
	return get_fast_S(f, R, nu, Sigma_f)

def harmonic(a, b):
	return 2*a*b/(a+b)


def plot_dx(phi, tag="T"):
	phi_x = (phi[1:]-phi[:-1])/(dx)
	phi_x = np.insert(phi_x, 0, 0)
	plt.plot(x[start:], phi_x[start:], label=r"$\frac{d}{dx} \phi_{%s}$" % tag)



# Main logic functions

In [ ]:
def get_A_operator(dx, D_f, D_r, Sigma_af, Sigma_ar):
	Ds = np.array([D_r]*R+[D_f]*N+[D_r]*R, dtype=float)
	Sigmas = np.array([Sigma_ar]*R+[Sigma_af]*N+[Sigma_ar]*R, dtype=float)
	
	diags_data = [
			np.ones(N-1+2*R) * -np.concat((Ds[:N//2+R],Ds[N//2+R+1:])) / dx**2,
			-2*np.ones(N+2*R) * -Ds / dx**2,
			np.ones(N-1+2*R) * -np.concat((Ds[:N//2+R],Ds[N//2+R+1:])) / dx**2
	]
	
	offsets = [-1, 0, 1]

	A = diags(diags_data, offsets, format="csc") 
	combined_diag = Sigmas*np.ones(N+2*R)
	A = A + diags([combined_diag], [0], shape=(N+2*R,N+2*R))

	# Continuity of current at R & N-R (Neumann)
	D_left = D_r
	D_right = D_f
	D_interface = 2*D_left*D_right / (D_left + D_right)

	i = R-1

	# row for node i
	#A[i, i-1] = -D_left / dx**2
	A[i, i]   = (D_left + D_interface)/dx**2 + Sigma_ar
	A[i, i+1] = -D_interface / dx**2

	# row for node i+1
	A[i+1, i]   = -D_interface / dx**2
	A[i+1, i+1] = (D_interface + D_right)/dx**2 + Sigma_af
	#A[i+1, i+2] = -D_right / dx**2

	D_left = D_f
	D_right = D_r
	D_interface = 2*D_left*D_right / (D_left + D_right)

	i = -R-1

	# row for node i
	#A[i, i-1] = -D_left / dx**2
	A[i, i]   = (D_left + D_interface)/dx**2 + Sigma_af
	A[i, i+1] = -D_interface / dx**2

	# row for node i+1
	A[i+1, i]   = -D_interface / dx**2
	A[i+1, i+1] = (D_interface + D_right)/dx**2 + Sigma_ar
	
	# Zero flux at extrapolated boundaries
	# Left boundary
	A[0,:] = 0
	A[0,0] = 1 + D_r*2.13/dx
	A[0,1] = -D_r*2.13/dx
	# Right boundary
	A[-1,:] = 0
	A[-1,-1] = 1 + D_r*2.13/dx
	A[-1,-2] = -D_r*2.13/dx

	return A

def calculate_flux(N, R, L, nu, Sigma_f, Sigma_sFT, Sigma_afT, Sigma_afF, Sigma_arT, Sigma_arF, D_fT, D_fF, D_rT, D_rF):
  
	k_eff	= 1
	x = np.linspace(0, L, N+2*R)
	dx = (N+2*R)/L
	
	AF = get_A_operator(dx, D_fF, D_rF, Sigma_afF, Sigma_arF)
	AT = get_A_operator(dx, D_fT, D_rT, Sigma_afT, Sigma_arT)

	phiT = np.ones(len(x))
	phiF = np.ones(len(x))
	SF = np.ones(len(x))

	epsilon = 1
	k_eff_old = 0
	max_iter = 40
	i = 0
	k_history = [k_eff]

	while epsilon > 0.001 and i < max_iter:
		i += 1

		SF_old = SF
		bF = SF_old / k_eff
		phiF = spsolve(AF, bF) #phi_f^(n+1)
		
		ST = get_thermal_S(phiF, R=R, Sigma_sFT=Sigma_sFT) # scattering source
		bT = ST
		phiT = spsolve(AT, bT) #phi_t^(n+1)

		max = np.max([phiT, phiF])
		phiT /= max
		phiF /= max

		SF = get_fast_S(phiT, R=R, nu=nu, Sigma_f=Sigma_f)

		k_eff = np.sum(SF) / np.sum(SF_old)
		k_history += [k_eff]

		epsilon = np.abs(k_eff-k_eff_old)
		k_eff_old = k_eff
		
	print(f"Finished in {i}/{max_iter} tries, k_eff: {k_eff:.5f}, eps: {epsilon:.5f}")

	return phiF, phiT, k_eff, k_history


def calculate_single_flux(N, R, L, Sigma_afT, Sigma_arT, D_fT, D_rT, nu=1, Sigma_f=1):
	x = np.linspace(0, L, N+2*R)
	dx = (N+2*R)/L
	
	phi = np.ones(len(x))
	S_simple = get_S(phi, R=R, nu=nu, Sigma_f=Sigma_f)
	A_simple = get_A_operator(dx, D_fT, D_rT, Sigma_afT, Sigma_arT)
	phi = iterate_sol(phi, A_simple, S_simple, 0, R=R, Sigma_f=Sigma_f)
	return phi

# Simulation constants

In [ ]:
nu_nom = 2.43 # just a constant amplitude

# Amount of U atoms in 20% enriched fuel?
N_235 = 9.79*10e-3
N_238 = 3.86*10e-2

Sigma_f_nom = 585 * N_235 # just a constant amplitude

Sigma_sFT_nom = 0.0419 # dominated by moderator

Sigma_afT_nom = 0.025 #680.8*N_235 + 7.59*N_238
Sigma_afF_nom = 0.005 #1.65*N_235 + 0.225*N_238

Sigma_arT_nom = 0.006 #0.0197
Sigma_arF_nom = 0.001 #0

D_fT_nom = 0.3668*1.55**2 # D = Sigma_a * L^2
D_fF_nom = 3 #1/(3*(6.8*N_235+6.9*N_238)) # guess
D_rT_nom = 0.16 #*7 # the closer to D_fT, the less discontinuous
D_rF_nom = 1.13

# Simulation run

In [ ]:
nu = nu_nom
Sigma_f = Sigma_f_nom
Sigma_sFT = Sigma_sFT_nom

Sigma_afT = Sigma_afT_nom
Sigma_afF = Sigma_afF_nom

Sigma_arT = Sigma_arT_nom
Sigma_arF = Sigma_arF_nom

D_fT = D_fT_nom
D_fF = D_fF_nom
D_rT = D_rT_nom
D_rF = D_rF_nom

R = 150
N = 500
L = 800 

x = np.linspace(0, L, N+2*R)
dx = (N+2*R)/L

# Simulation logic: coupled fluxes calculation
phiF, phiT, k_eff, k_history = calculate_flux(N, R, L, nu, Sigma_f, Sigma_sFT, Sigma_afT, Sigma_afF, Sigma_arT, Sigma_arF, D_fT, D_fF, D_rT, D_rF)
phi_simple = calculate_single_flux(N, R, L, Sigma_afT/20, Sigma_arT, D_fT, D_rT) # Sigma_afT initially is too high for a single flux
	
print("Final coupled k_eff:", k_eff)

start = 0 #R+N//2 # remove the comment to start from the middle
plt.plot(x, phi_simple, label=r"Individual $\phi_T$", linestyle='--')
plt.plot(x, phiT, label="Coupled $\phi_T$")
plt.plot(x, phiF, label="Coupled $\phi_1$")

if start < R/(N+2*R)*L:
	plt.vlines(R/(N+2*R)*L, 0, 1, linestyle=':')
plt.vlines((N+R)/(N+2*R)*L, 0, 1, linestyle=':')

plot_dx(phiT*2, "T")
plot_dx(phiF*20, "F")

plt.legend()
plt.xlim(left=start, right=L)
plt.xlabel("Cell number (#)")
plt.ylabel("Flux intensity (a.u.)")

plt.show()

# Simulation Playground
Click on the sliders instead of dragging for better performance!

In [ ]:
%matplotlib qt
from matplotlib.widgets import Slider

fig, (ax, ax_empty) = plt.subplots(ncols=2, width_ratios=[4, 5], figsize=(10,4))
phiF, phiT, _, _ = calculate_flux(N, R, L, nu, Sigma_f, Sigma_sFT, Sigma_afT, Sigma_afF, Sigma_arT, Sigma_arF, D_fT, D_fF, D_rT, D_rF)

ax_empty.axis('off')	

ax.set_xlabel("Cell number (#)")
ax.set_ylabel("Flux intensity (a.u.)")
line, = ax.plot(x, phiT, label="flux T")
line2, = ax.plot(x, phiF, label="flux F")
plt.tight_layout()

sliders = {}

ax_h = 0.03
bottom0 = 0.25

def make_slider(name, val_nom, axpos, label):
	val_min = 0.2 * val_nom
	val_max = 5.0 * val_nom
	ax = plt.axes([0.48, axpos, 0.3, 0.04])
	sliders[name] = Slider(ax, label, val_min, val_max, valinit=val_nom)
	sliders[name].label.set_x(1.25)         # push it past the right edge
	sliders[name].label.set_horizontalalignment('left')


make_slider('D_rF'      , D_rF_nom      , bottom0 + 0,    'D_r,F   (reflector)')
make_slider('D_rT'      , D_rT_nom      , bottom0 + 0.05, 'D_r,T   (reflector)')
make_slider('Sigma_arF' , Sigma_arF_nom , bottom0 + 0.10, 'Σ_a,r,F (reflector)')
make_slider('Sigma_arT' , Sigma_arT_nom , bottom0 + 0.15, 'Σ_a,r,T (reflector)')
make_slider('Sigma_sFT' , Sigma_sFT_nom , bottom0 + 0.20, 'Σ_s1    (scattering)')
make_slider('D_fF'      , D_fF_nom      , bottom0 + 0.25, 'D_f,F   (fuel)')
make_slider('D_fT'      , D_fT_nom      , bottom0 + 0.30, 'D_f,T   (fuel)')
make_slider('Sigma_afF' , Sigma_afF_nom , bottom0 + 0.35, 'Σ_f,a,F (fuel)')
make_slider('Sigma_afT' , Sigma_afT_nom , bottom0 + 0.40, 'Σ_f,a,T (fuel)')
make_slider('Sigma_f'   , Sigma_f_nom   , bottom0 + 0.45, 'Σ_f')
make_slider('nu'    , nu_nom    , bottom0 + 0.50, 'ν')

Sigma_f   = sliders['Sigma_f'   ].val
nu     =	sliders['nu'    ].val
Sigma_sFT = sliders['Sigma_sFT' ].val
Sigma_afT = sliders['Sigma_afT' ].val
Sigma_afF = sliders['Sigma_afF' ].val
Sigma_arT = sliders['Sigma_arT' ].val
Sigma_arF = sliders['Sigma_arF' ].val
D_fT      = sliders['D_fT'      ].val
D_fF      = sliders['D_fF'      ].val
D_rT      = sliders['D_rT'      ].val
D_rF      = sliders['D_rF'      ].val


phiF, phiT, _, _ = calculate_flux(N, R, L, nu, Sigma_f, Sigma_sFT, Sigma_afT, Sigma_afF, Sigma_arT, Sigma_arF, D_fT, D_fF, D_rT, D_rF)
	
line.set_xdata(x)
line.set_ydata(phiT)
line.set_label("$\phi_T$")
line2.set_xdata(x)
line2.set_ydata(phiF)
line2.set_label("$\phi_F$")
ax.legend()
fig.canvas.draw_idle()

def update(val):
	Sigma_f   = sliders['Sigma_f'   ].val
	nu     = 	sliders['nu'    ].val
	Sigma_sFT = sliders['Sigma_sFT' ].val
	Sigma_afT = sliders['Sigma_afT' ].val
	Sigma_afF = sliders['Sigma_afF' ].val
	Sigma_arT = sliders['Sigma_arT' ].val
	Sigma_arF = sliders['Sigma_arF' ].val
	D_fT      = sliders['D_fT'      ].val
	D_fF      = sliders['D_fF'      ].val
	D_rT      = sliders['D_rT'      ].val
	D_rF      = sliders['D_rF'      ].val

	phiF, phiT, k_eff, _ = calculate_flux(N, R, L, nu, Sigma_f, Sigma_sFT, Sigma_afT, Sigma_afF, Sigma_arT, Sigma_arF, D_fT, D_fF, D_rT, D_rF)
	
	#print("Final coupled k_eff:", k_eff)

	line.set_xdata(x)
	line.set_ydata(phiT)
	line2.set_xdata(x)
	line2.set_ydata(phiF)
	fig.canvas.draw_idle()
	print("Final coupled k_eff:", k_eff)

for s in sliders.values():
	s.on_changed(update)